### Data extractor 

In [1]:
import json
import zipfile
import pandas as pd
import matplotlib.pyplot as plt
from plotly.subplots import make_subplots
from plotly.colors import hex_to_rgb
import plotly.graph_objects as go
import plotly.express as px
import tqdm
import ipywidgets as widgets
from IPython.display import display


In [2]:
def parse_zip_experiments(zip_filepath):
    data = []
    
    with zipfile.ZipFile(zip_filepath, 'r') as archive:
        for filename in tqdm.tqdm(archive.namelist()):
            if filename.endswith('.json'):
                with archive.open(filename) as f:
                    try:
                        exp = json.load(f)
                        
                        params = exp.get("parameters", {})
                        num_agents = params.get("numAgents")
                        num_objects = params.get("numObjects")
                        ratio_random = params.get("ratioRandom", 0) 
                        seed_val = params.get("seed", "Inconnu")
                        
                        policy_val = params.get("selectedPolicy", "Standard")

                        results = exp.get("results", {})
                        
                        solver = results.get("solver", {})
                        solver_time = solver.get("timeUs", 0)
                        solver_score = solver.get("score", 0)
                        solver_metrics = solver.get("metrics", {})
                        
                        mcts = results.get("mcts", {})
                        mcts_tries = mcts.get("tries", [])
                        
                        mcts_total_times = []
                        mcts_final_scores = []
                        
                        mcts_metrics_counts = {
                            "ParetoOptimal": 0, "Prop": 0, 
                            "EF": 0, "EFX": 0, "EF1": 0
                        }
                        
                        for t in mcts_tries:
                            steps = t.get("steps", [])
                            if steps:
                                mcts_total_times.append(sum(step.get("stepTimeUs", 0) for step in steps))
                                mcts_final_scores.append(steps[-1].get("score", 0))
                                final_metrics = steps[-1].get("metrics", {})
                                for metric in mcts_metrics_counts.keys():
                                    if final_metrics.get(metric, False):
                                        mcts_metrics_counts[metric] += 1
                                        
                        # --- NOUVEAU : Calcul des moyennes, mins et maxs ---
                        if mcts_total_times:
                            num_tries = len(mcts_total_times)
                            row_data = {
                                "Agents": num_agents,
                                "Objets": num_objects,
                                "Ratio": ratio_random,
                                "Seed" : seed_val,
                                "Policy": policy_val,
                                "Solveur_Temps_us": solver_time,
                                "Solveur_Score": solver_score,
                                "MCTS_Temps_Moyen_us": sum(mcts_total_times) / num_tries,
                                "MCTS_Temps_Min_us": min(mcts_total_times),
                                "MCTS_Temps_Max_us": max(mcts_total_times),
                                "MCTS_Score_Moyen": sum(mcts_final_scores) / num_tries,
                                "MCTS_Score_Min": min(mcts_final_scores),
                                "MCTS_Score_Max": max(mcts_final_scores)
                            }
                            for metric in mcts_metrics_counts.keys():
                                row_data[f"Solveur_{metric}"] = int(solver_metrics.get(metric, False))
                                row_data[f"MCTS_{metric}"] = mcts_metrics_counts[metric] / num_tries
                            data.append(row_data)
                    except Exception as e:
                        print(f"Erreur lors de la lecture de {filename} : {e}")

    return pd.DataFrame(data)

In [3]:
def plot_comparisons(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On identifie combien de configurations d'agents différentes tu as
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    # Pour chaque nombre d'agents, on dessine un graphique séparé
    for agents in configurations_agents:
        # On filtre pour ne garder que les données de ce nombre d'agents
        df_filtre = df[df['Agents'] == agents]
        
        # On fait la moyenne par nombre d'objets
        df_grouped = df_filtre.groupby('Objets').mean().reset_index()

        # Si on n'a qu'un seul point (un seul nombre d'objets testé pour ce nombre d'agents), on le signale
        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations d'objets pour tracer une courbe pour {agents} agents.")
            continue

        # Création de la figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Analyse pour {agents} Agents", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Objets) ---
        ax1.plot(df_grouped['Objets'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Objets'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps d'exécution")
        ax1.set_xlabel("Nombre d'Objets")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log') # Conserve l'échelle logarithmique pour les temps exponentiels
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Objets) ---
        ax2.plot(df_grouped['Objets'], df_grouped['Solveur_Score'], marker='o', label='Solveur', color='red', linewidth=2)
        ax2.plot(df_grouped['Objets'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores")
        ax2.set_xlabel("Nombre d'Objets")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [4]:
def plot_comparisons_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Options des widgets
    agents_options = ['Tous'] + sorted(df['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df['Objets'].unique().tolist())
    seed_options = ['Tous'] + sorted(df['Seed'].unique().tolist()) if 'Seed' in df.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df['Policy'].unique().tolist()) if 'Policy' in df.columns else ["Non défini"]

    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value=agents_options[1] if len(agents_options)>1 else 'Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            ag, ob, sd, pol = dropdown_agents.value, dropdown_objets.value, dropdown_seed.value, dropdown_policy.value
            
            df_filtre = df.copy()
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]

            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée.")
                return

            # Axe X
            if ag != 'Tous' and ob == 'Tous':
                x_col, x_title = 'Objets', "Nombre d'Objets"
            elif ag == 'Tous' and ob != 'Tous':
                x_col, x_title = 'Agents', "Nombre d'Agents"
            else:
                x_col, x_title = 'Objets', "Nombre d'Objets"

            fig = make_subplots(rows=1, cols=2, subplot_titles=("Temps d'exécution", "Scores Absolus"))

            # Courbe du Solveur (Moyenne globale sur la config pour éviter la surcharge)
            df_solveur = df_filtre.groupby(x_col).mean(numeric_only=True).reset_index()
            fig.add_trace(go.Scatter(x=df_solveur[x_col], y=df_solveur['Solveur_Temps_us'], mode='lines+markers', name='Solveur (Temps)', line=dict(color='red', width=3)), row=1, col=1)
            fig.add_trace(go.Scatter(x=df_solveur[x_col], y=df_solveur['Solveur_Score'], mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot', width=3)), row=1, col=2)

            # Courbes MCTS : une par politique si 'Tous' est sélectionné
            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                for p in sorted(df_filtre['Policy'].unique()):
                    df_p = df_filtre[df_filtre['Policy'] == p].groupby(x_col).mean(numeric_only=True).reset_index()
                    fig.add_trace(go.Scatter(x=df_p[x_col], y=df_p['MCTS_Temps_Moyen_us'], mode='lines+markers', name=f'MCTS {p} (Temps)'), row=1, col=1)
                    fig.add_trace(go.Scatter(x=df_p[x_col], y=df_p['MCTS_Score_Moyen'], mode='lines+markers', name=f'MCTS {p} (Score)', line=dict(dash='dot')), row=1, col=2)
            else:
                df_mcts = df_filtre.groupby(x_col).mean(numeric_only=True).reset_index()
                fig.add_trace(go.Scatter(x=df_mcts[x_col], y=df_mcts['MCTS_Temps_Moyen_us'], mode='lines+markers', name=f'MCTS {pol} (Temps)', line=dict(color='blue')), row=1, col=1)
                fig.add_trace(go.Scatter(x=df_mcts[x_col], y=df_mcts['MCTS_Score_Moyen'], mode='lines+markers', name=f'MCTS {pol} (Score)', line=dict(color='blue', dash='dot')), row=1, col=2)

            titre = f"Comparaison MCTS vs Solveur — Agents: {ag} | Objets: {ob} | Graine: {sd} | Pol: {pol}"
            fig.update_layout(title_text=titre, hovermode="x unified", template="plotly_white")
            type_x = 'category' if x_col in ['Policy', 'Seed'] else '-'
            fig.update_xaxes(title_text=x_title, type=type_x, row=1, col=1).update_xaxes(title_text=x_title, type=type_x, row=1, col=2)
            fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1).update_yaxes(title_text="Score", row=1, col=2)
            fig.show()

    for w in [dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy]: w.observe(update_plot, names='value')
    display(ui, out)
    update_plot()

In [5]:
def plot_impact_ratio(df):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On sépare les graphiques par nombre d'agents pour que ce soit lisible
    configurations_agents = df['Agents'].unique()
    configurations_agents.sort()

    for agents in configurations_agents:
        df_filtre = df[df['Agents'] == agents]
        
        # On groupe par Ratio pour voir son impact global (moyenne sur tous les objets et seeds)
        df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

        if len(df_grouped) < 2:
            print(f"⚠️ Pas assez de variations de 'ratioRandom' pour tracer une courbe pour {agents} agents.")
            continue

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        fig.suptitle(f"Impact du Ratio Random ({agents} Agents)", fontsize=16, fontweight='bold')

        # --- Courbe 1 : Temps d'exécution (Axe X = Ratio) ---
        # Le solveur est tracé pour vérifier si le ratio impacte la génération du problème
        ax1.plot(df_grouped['Ratio'], df_grouped['Solveur_Temps_us'], marker='o', label='Solveur', color='red', linewidth=2)
        ax1.plot(df_grouped['Ratio'], df_grouped['MCTS_Temps_Moyen_us'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax1.set_title("Évolution du Temps selon le Ratio")
        ax1.set_xlabel("Ratio Random")
        ax1.set_ylabel("Temps (microsecondes)")
        ax1.set_yscale('log')
        ax1.legend()
        ax1.grid(True, which="both", ls="--")

        # --- Courbe 2 : Score (Axe X = Ratio) ---
        ax2.plot(df_grouped['Ratio'], df_grouped['Solveur_Score'], marker='o', label='Solveur (Optimum)', color='red', linewidth=2)
        ax2.plot(df_grouped['Ratio'], df_grouped['MCTS_Score_Moyen'], marker='s', label='MCTS (Moyenne)', color='blue', linewidth=2)
        ax2.set_title("Évolution des Scores selon le Ratio")
        ax2.set_xlabel("Ratio Random")
        ax2.set_ylabel("Score")
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()
        plt.show()

In [6]:
def plot_impact_ratio_interactive(df):
    if df.empty: return
    
    agents_options = ['Tous'] + sorted(df['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df['Objets'].unique().tolist())
    seed_options = ['Tous'] + sorted(df['Seed'].unique().tolist()) if 'Seed' in df.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df['Policy'].unique().tolist()) if 'Policy' in df.columns else ["Non défini"]

    dropdown_agents = widgets.Dropdown(options=agents_options, value=agents_options[1] if len(agents_options)>1 else 'Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            ag, ob, sd, pol = dropdown_agents.value, dropdown_objets.value, dropdown_seed.value, dropdown_policy.value
            
            df_filtre = df.copy()
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]

            if df_filtre.empty or 'Ratio' not in df_filtre.columns: return

            fig = make_subplots(rows=1, cols=2, subplot_titles=("Temps selon le Ratio", "Score selon le Ratio"))

            # Base Solveur
            df_solveur = df_filtre.groupby('Ratio').mean(numeric_only=True).reset_index().sort_values('Ratio')
            fig.add_trace(go.Scatter(x=df_solveur['Ratio'], y=df_solveur['Solveur_Temps_us'], mode='lines+markers', name='Solveur (Temps)', line=dict(color='red', width=3)), row=1, col=1)
            fig.add_trace(go.Scatter(x=df_solveur['Ratio'], y=df_solveur['Solveur_Score'], mode='lines+markers', name='Solveur (Score)', line=dict(color='red', dash='dot', width=3)), row=1, col=2)

            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                for p in sorted(df_filtre['Policy'].unique()):
                    df_p = df_filtre[df_filtre['Policy'] == p].groupby('Ratio').mean(numeric_only=True).reset_index().sort_values('Ratio')
                    fig.add_trace(go.Scatter(x=df_p['Ratio'], y=df_p['MCTS_Temps_Moyen_us'], mode='lines+markers', name=f'MCTS {p} (Temps)'), row=1, col=1)
                    fig.add_trace(go.Scatter(x=df_p['Ratio'], y=df_p['MCTS_Score_Moyen'], mode='lines+markers', name=f'MCTS {p} (Score)', line=dict(dash='dot')), row=1, col=2)
            else:
                df_mcts = df_filtre.groupby('Ratio').mean(numeric_only=True).reset_index().sort_values('Ratio')
                fig.add_trace(go.Scatter(x=df_mcts['Ratio'], y=df_mcts['MCTS_Temps_Moyen_us'], mode='lines+markers', name=f'MCTS {pol} (Temps)', line=dict(color='blue')), row=1, col=1)
                fig.add_trace(go.Scatter(x=df_mcts['Ratio'], y=df_mcts['MCTS_Score_Moyen'], mode='lines+markers', name=f'MCTS {pol} (Score)', line=dict(color='blue', dash='dot')), row=1, col=2)

            fig.update_layout(title_text=f"Impact du Ratio Random — Pol: {pol}", hovermode="x unified", template="plotly_white")
            fig.update_xaxes(title_text="Ratio Random", row=1, col=1).update_xaxes(title_text="Ratio Random", row=1, col=2)
            fig.update_yaxes(title_text="Temps (µs)", type="log", row=1, col=1).update_yaxes(title_text="Score", row=1, col=2)
            fig.show()

    for w in [dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy]: w.observe(update_plot, names='value')
    display(ui, out)
    update_plot()

In [7]:
def plot_ratio_impact_on_score(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée dans le ZIP.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_score = df[df['Solveur_Score'] > 0].copy()
    
    # Calcul du pourcentage d'optimalité
    df_score['Optimalite_MCTS_%'] = (df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100

    configurations_agents = df_score['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Tout sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(8, 5)) 

        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
        plt.title("Impact du Ratio sur la qualité du score (Toutes configurations)", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Qualité de la solution (%)")
        plt.ylim(0, 105) 
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par nombre d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_score[df_score['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                print(f"⚠️ Pas assez de variations de 'ratioRandom' pour {agents} agents.")
                continue

            plt.figure(figsize=(7, 4))
            
            plt.plot(df_grouped['Ratio'], df_grouped['Optimalite_MCTS_%'], 
                     marker='o', color='purple', linewidth=2, label='Score MCTS')
            plt.axhline(y=100, color='red', linestyle='--', linewidth=2, label='Optimum (Solveur = 100%)')
            
            plt.title(f"Impact du Ratio sur la qualité du score ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Qualité de la solution (%)")
            plt.ylim(0, 105) 
            plt.legend()
            plt.grid(True, which="both", ls="--")
            plt.tight_layout()
            plt.show()

In [8]:
def plot_ratio_impact_on_score_interactive(df):
    if df.empty: return
    df_score = df[df['Solveur_Score'] > 0].copy()
    df_score['Optimalite_MCTS_%'] = ((df_score['MCTS_Score_Moyen'] / df_score['Solveur_Score']) * 100).clip(upper=100)

    agents_options = ['Tous'] + sorted(df_score['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_score['Objets'].unique().tolist())
    seed_options = ['Tous'] + sorted(df_score['Seed'].unique().tolist()) if 'Seed' in df_score.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df_score['Policy'].unique().tolist()) if 'Policy' in df_score.columns else ["Non défini"]

    dropdown_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})

    ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            ag, ob, sd, pol = dropdown_agents.value, dropdown_objets.value, dropdown_seed.value, dropdown_policy.value

            df_filtre = df_score.copy()
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]

            if df_filtre.empty or 'Ratio' not in df_filtre.columns: return

            # Regroupement intelligent incluant 'Policy' si demandé
            group_cols = ['Agents', 'Ratio']
            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                group_cols.append('Policy')

            df_grouped = df_filtre.groupby(group_cols).mean(numeric_only=True).reset_index().sort_values('Ratio')
            
            # Création d'une variable pour distinguer les courbes dans la légende
            if pol == 'Tous' and 'Policy' in df_grouped.columns:
                df_grouped['Légende'] = df_grouped['Agents'].astype(str) + " Ag | " + df_grouped['Policy']
            else:
                df_grouped['Légende'] = df_grouped['Agents'].astype(str) + " Agents"

            fig = px.line(
                df_grouped, x="Ratio", y="Optimalite_MCTS_%",
                color="Légende", markers=True,
                title=f"Impact du Ratio sur l'Optimalité — Pol: {pol}",
                labels={"Ratio": "Ratio Random", "Optimalite_MCTS_%": "Qualité de la solution (%)"}
            )
            fig.add_hline(y=100, line_dash="dash", line_color="red")
            fig.update_layout(yaxis=dict(range=[max(0, df_grouped['Optimalite_MCTS_%'].min() - 5), 102]), hovermode="x unified", template="plotly_white")
            fig.show()

    for w in [dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy]: w.observe(update_plot, names='value')
    display(ui, out)
    update_plot()

In [9]:
def plot_erreur_normalisee(df, combine_agents=True):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # On copie le DataFrame et on s'assure de ne pas diviser par zéro
    df_safe = df[df['Solveur_Score'] > 0].copy()

    # Calcul de l'erreur normalisée globale en pourcentage pour chaque ligne
    df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

    configurations_agents = df_safe['Agents'].unique()
    configurations_agents.sort()

    # --- CAS 1 : Toutes les courbes sur le même graphique ---
    if combine_agents:
        plt.figure(figsize=(10, 6))

        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
                
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', linewidth=2, label=f'MCTS ({agents} Agents)')

        plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

        plt.title("Erreur Normalisée globale du MCTS selon le Ratio", fontsize=14, fontweight='bold')
        plt.xlabel("Ratio Random")
        plt.ylabel("Erreur par rapport au Solveur (%)")
        
        erreur_max = df_safe['Erreur_%'].max()
        plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
        
        plt.grid(True, which="both", ls="--")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # --- CAS 2 : Un graphique séparé par configuration d'agents ---
    else:
        for agents in configurations_agents:
            df_filtre = df_safe[df_safe['Agents'] == agents]
            df_grouped = df_filtre.groupby('Ratio').mean().reset_index()

            if len(df_grouped) < 2:
                continue
            
            plt.figure(figsize=(7, 4))
            
            # Utilisation d'une couleur neutre si c'est affiché seul
            plt.plot(df_grouped['Ratio'], df_grouped['Erreur_%'], 
                     marker='o', color='purple', linewidth=2, label='Erreur MCTS')

            plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Optimum (Erreur = 0%)')

            plt.title(f"Erreur Normalisée du MCTS selon le Ratio ({agents} Agents)", fontsize=14, fontweight='bold')
            plt.xlabel("Ratio Random")
            plt.ylabel("Erreur par rapport au Solveur (%)")
            
            # Ajustement dynamique du zoom pour ce graphique spécifique
            erreur_max = df_filtre['Erreur_%'].max()
            plt.ylim(-1, erreur_max + 2 if pd.notna(erreur_max) else 10)
            
            plt.grid(True, which="both", ls="--")
            plt.legend()
            plt.tight_layout()
            plt.show()
         

In [10]:
def plot_erreur_normalisee_interactive(df):
    if df.empty: return
    df_safe = df[df['Solveur_Score'] > 0].copy()
    df_safe['Erreur_%'] = ((df_safe['Solveur_Score'] - df_safe['MCTS_Score_Moyen']) / df_safe['Solveur_Score']) * 100

    agents_options = ['Tous'] + sorted(df_safe['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_safe['Objets'].unique().tolist())
    seed_options = ['Tous'] + sorted(df_safe['Seed'].unique().tolist()) if 'Seed' in df_safe.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df_safe['Policy'].unique().tolist()) if 'Policy' in df_safe.columns else ["Non défini"]

    dropdown_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})

    ui = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            ag, ob, sd, pol = dropdown_agents.value, dropdown_objets.value, dropdown_seed.value, dropdown_policy.value

            df_filtre = df_safe.copy()
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]

            if df_filtre.empty or 'Ratio' not in df_filtre.columns: return

            group_cols = ['Agents', 'Ratio']
            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                group_cols.append('Policy')

            df_grouped = df_filtre.groupby(group_cols).mean(numeric_only=True).reset_index().sort_values('Ratio')
            
            if pol == 'Tous' and 'Policy' in df_grouped.columns:
                df_grouped['Légende'] = df_grouped['Agents'].astype(str) + " Ag | " + df_grouped['Policy']
            else:
                df_grouped['Légende'] = df_grouped['Agents'].astype(str) + " Agents"

            fig = px.line(
                df_grouped, x="Ratio", y="Erreur_%", color="Légende", markers=True,
                title=f"Impact du Ratio sur l'Erreur Normalisée — Pol: {pol}",
                labels={"Ratio": "Ratio Random", "Erreur_%": "Erreur par rapport au Solveur (%)"}
            )
            fig.add_hline(y=0, line_dash="dash", line_color="green")
            fig.update_layout(hovermode="x unified", template="plotly_white")
            fig.show()

    for w in [dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy]: w.observe(update_plot, names='value')
    display(ui, out)
    update_plot()

In [11]:
def plot_metrics_comparison_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return
        
    df_base = df.copy()
            
    # 1. Options des widgets
    agents_options = ['Tous'] + sorted(df_base['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_base['Objets'].unique().tolist())
    seed_options = ['Tous'] + sorted(df_base['Seed'].unique().tolist()) if 'Seed' in df_base.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df_base['Policy'].unique().tolist()) if 'Policy' in df_base.columns else ["Non défini"]
    
    if 'Ratio' in df_base.columns:
        ratio_options = ['Tous'] + sorted(df_base['Ratio'].unique().tolist())
    else:
        ratio_options = ["Non défini"]

    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    slider_ratio = widgets.SelectionSlider(
        options=ratio_options,
        value=ratio_options[1] if len(ratio_options)>1 else 'Tous',
        description='🎲 Ratio:',
        continuous_update=False, 
        layout={'width': '400px'}
    )
    
    checkbox_filter_zeros = widgets.Checkbox(value=True, description="🚫 Exclure les scores du solveur = 0", layout={'width': 'max-content'})

    ui_top = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy, checkbox_filter_zeros])
    ui = widgets.VBox([ui_top, slider_ratio])
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            ag, ob, rat, sd, pol = dropdown_agents.value, dropdown_objets.value, slider_ratio.value, dropdown_seed.value, dropdown_policy.value
            filtrer_zeros = checkbox_filter_zeros.value

            # 3. Filtrage dynamique
            df_filtre = df_base.copy()
            
            if filtrer_zeros and 'Solveur_Score' in df_filtre.columns:
                df_filtre = df_filtre[df_filtre['Solveur_Score'] != 0]
                
            if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
            if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
            if rat != 'Tous' and rat != "Non défini": df_filtre = df_filtre[df_filtre['Ratio'] == rat]
            if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
            if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]

            if df_filtre.empty:
                print(f"⚠️ Aucune donnée trouvée pour cette sélection.")
                return

            metrics_list = ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]
            metrics_list = [m for m in metrics_list if f"Solveur_{m}" in df_filtre.columns]

            if not metrics_list:
                print("⚠️ Aucune colonne de métrique trouvée.")
                return

            # 4. Construction du graphique barmode='group'
            fig = go.Figure()
            
            # Trace du Solveur (Moyenne globale sur le filtre pour référence)
            solveur_means = [df_filtre[f"Solveur_{m}"].mean() * 100 for m in metrics_list]
            fig.add_trace(go.Bar(
                x=metrics_list, 
                y=solveur_means, 
                name='Solveur', 
                marker_color='red'
            ))

            # Traces MCTS
            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                # On ajoute une barre par politique existante dans le scope filtré
                for p in sorted(df_filtre['Policy'].unique()):
                    df_p = df_filtre[df_filtre['Policy'] == p]
                    mcts_means = [df_p[f"MCTS_{m}"].mean() * 100 for m in metrics_list]
                    fig.add_trace(go.Bar(
                        x=metrics_list, 
                        y=mcts_means, 
                        name=f'MCTS — {p}'
                    ))
            else:
                # Une seule barre pour la politique sélectionnée (ou 'Tous' s'il n'y a pas la colonne Policy)
                mcts_means = [df_filtre[f"MCTS_{m}"].mean() * 100 for m in metrics_list]
                fig.add_trace(go.Bar(
                    x=metrics_list, 
                    y=mcts_means, 
                    name=f'MCTS — {pol}', 
                    marker_color='blue'
                ))

            # 5. Mise en forme
            titre = f"Respect des Métriques — Pol: {pol} | Ag: {ag} | Ob: {ob} | Ratio: {rat} | Graine: {sd}"
            fig.update_layout(
                title_text=titre,
                xaxis_title="Métriques",
                yaxis_title="Taux de réussite (%)",
                barmode='group',
                template="plotly_white",
                yaxis=dict(range=[0, 105]),
                margin=dict(t=60, b=40, l=40, r=40)
            )
            fig.show()

    # 6. Liaisons
    dropdown_agents.observe(update_plot, names='value')
    dropdown_objets.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    dropdown_seed.observe(update_plot, names='value')
    dropdown_policy.observe(update_plot, names='value')
    checkbox_filter_zeros.observe(update_plot, names='value')
    
    display(ui, out)
    update_plot()

In [12]:
def widget_variance_mcts_normalise(df):
    """
    Crée un widget interactif affichant la variance du MCTS en POURCENTAGE 
    du score optimal, reliant les points d'une même politique par des pointillés.
    """
    if df.empty:
        print("Le DataFrame est vide.")
        return
        
    # Sécurité : on écarte les instances où le score du solveur est 0 pour éviter la division par zéro
    df_safe = df[df['Solveur_Score'] > 0].copy()
    
    # 1. Identifier les valeurs uniques pour les filtres
    agents_options = ['Tous'] + sorted(df_safe['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_safe['Objets'].unique().tolist())
    ratio_options = ['Tous'] + sorted(df_safe['Ratio'].unique().tolist()) if 'Ratio' in df_safe.columns else ["Non défini"]
    seed_options = ['Tous'] + sorted(df_safe['Seed'].unique().tolist()) if 'Seed' in df_safe.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df_safe['Policy'].unique().tolist()) if 'Policy' in df_safe.columns else ["Non défini"]
        
    # 2. Création des widgets
    dropdown_agents = widgets.Dropdown(options=agents_options, value=agents_options[1] if len(agents_options)>1 else 'Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dropdown_objets = widgets.Dropdown(options=objets_options, value=objets_options[1] if len(objets_options)>1 else 'Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dropdown_seed = widgets.Dropdown(options=seed_options, value='Tous', description='🌱 Graine:', layout={'width': 'max-content'})
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    slider_ratio = widgets.SelectionSlider(
        options=ratio_options,
        value=ratio_options[1] if len(ratio_options)>1 else 'Tous',
        description='🎲 Ratio:',
        disabled=False,
        continuous_update=False, 
        orientation='horizontal',
        readout=True,
        layout={'width': '400px'}
    )
    
    ui_top = widgets.HBox([dropdown_agents, dropdown_objets, dropdown_seed, dropdown_policy])
    ui = widgets.VBox([ui_top, slider_ratio])
    
    out = widgets.Output()
    
    def update_plot(change=None):
        out.clear_output(wait=True)
        
        ag = dropdown_agents.value
        ob = dropdown_objets.value
        sd = dropdown_seed.value
        rat = slider_ratio.value
        pol = dropdown_policy.value
        
        # 3. Filtrage dynamique
        df_filtre = df_safe.copy()
        
        if ag != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == ag]
        if ob != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == ob]
        if sd != 'Tous' and sd != "Non défini": df_filtre = df_filtre[df_filtre['Seed'] == sd]
        if rat != 'Tous' and rat != "Non défini": df_filtre = df_filtre[df_filtre['Ratio'] == rat]
        if pol != 'Tous' and pol != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == pol]
            
        with out:
            if df_filtre.empty:
                print(f"⚠️ Aucune instance trouvée pour cette configuration.")
                return
                
            # Calcul des pourcentages d'optimalité
            df_filtre['Opti_Moyen_%'] = (df_filtre['MCTS_Score_Moyen'] / df_filtre['Solveur_Score']) * 100
            df_filtre['Opti_Min_%'] = (df_filtre['MCTS_Score_Min'] / df_filtre['Solveur_Score']) * 100
            df_filtre['Opti_Max_%'] = (df_filtre['MCTS_Score_Max'] / df_filtre['Solveur_Score']) * 100
            
            # Détermination d'un index d'instance unique et stable pour aligner les courbes sur l'axe X
            instance_cols = ['Agents', 'Objets', 'Ratio', 'Seed']
            instance_cols = [c for c in instance_cols if c in df_filtre.columns]
            
            df_instances = df_filtre[instance_cols].drop_duplicates().sort_values('Seed' if 'Seed' in df_filtre.columns else instance_cols).reset_index(drop=True)
            df_instances['Instance_Idx'] = df_instances.index
            
            df_filtre = df_filtre.merge(df_instances, on=instance_cols)
            
            fig = go.Figure()
            
            # A. Ligne de l'optimum de référence (100%)
            fig.add_hline(
                y=100, 
                line_dash="dash", 
                line_color="red", 
                annotation_text="Score Optimal (Solveur) = 100%", 
                annotation_position="bottom right"
            )
            
            # B. Points et lignes MCTS en pourcentage avec barres d'erreur
            if pol == 'Tous' and 'Policy' in df_filtre.columns:
                # Affichage de toutes les politiques (une courbe pointillée par politique)
                for p in sorted(df_filtre['Policy'].unique()):
                    df_p = df_filtre[df_filtre['Policy'] == p].sort_values('Instance_Idx')
                    
                    hover_text = (
                        "<b>Instance n°%{x}</b><br>"
                        f"Politique : <b>{p}</b><br>"
                        "Score MCTS Moyen : %{customdata[0]:.2f} (%{y:.1f}% de l'optimum)<br>"
                        "Score Solveur absolu : %{customdata[1]:.2f}"
                    )
                    if sd == 'Tous' and 'Seed' in df_p.columns:
                        hover_text += "<br>🌱 Graine : %{customdata[2]}"
                        
                    custom_data = df_p[['MCTS_Score_Moyen', 'Solveur_Score', 'Seed']].values if 'Seed' in df_p.columns else df_p[['MCTS_Score_Moyen', 'Solveur_Score']].values
                    
                    fig.add_trace(go.Scatter(
                        x=df_p['Instance_Idx'],
                        y=df_p['Opti_Moyen_%'],
                        mode='lines+markers',                # Activé 'lines' en plus des 'markers'
                        line=dict(dash='dot', width=1.5),    # Applique le style pointillé
                        name=f'MCTS — {p}',
                        customdata=custom_data,
                        hovertemplate=hover_text,
                        error_y=dict(
                            type='data',
                            symmetric=False,
                            array=df_p['Opti_Max_%'] - df_p['Opti_Moyen_%'],
                            arrayminus=df_p['Opti_Moyen_%'] - df_p['Opti_Min_%'],
                            thickness=1,
                            width=3
                        )
                    ))
            else:
                # Affichage d'une seule politique sélectionnée
                df_single = df_filtre.sort_values('Instance_Idx')
                hover_text = (
                    "<b>Instance n°%{x}</b><br>"
                    f"Politique : <b>{pol}</b><br>"
                    "Score MCTS Moyen : %{customdata[0]:.2f} (%{y:.1f}% de l'optimum)<br>"
                    "Score Solveur absolu : %{customdata[1]:.2f}"
                )
                if sd == 'Tous' and 'Seed' in df_single.columns:
                    hover_text += "<br>🌱 Graine : %{customdata[2]}"
                    
                custom_data = df_single[['MCTS_Score_Moyen', 'Solveur_Score', 'Seed']].values if 'Seed' in df_single.columns else df_single[['MCTS_Score_Moyen', 'Solveur_Score']].values
                
                fig.add_trace(go.Scatter(
                    x=df_single['Instance_Idx'],
                    y=df_single['Opti_Moyen_%'],
                    mode='lines+markers',                    # Mode lignes + points
                    line=dict(color='blue', dash='dot', width=2),  # Ligne bleue en pointillés
                    name=f'MCTS — {pol}',
                    marker=dict(color='blue', size=8),
                    customdata=custom_data,
                    hovertemplate=hover_text,
                    error_y=dict(
                        type='data',
                        symmetric=False,
                        array=df_single['Opti_Max_%'] - df_single['Opti_Moyen_%'],
                        arrayminus=df_single['Opti_Moyen_%'] - df_single['Opti_Min_%'],
                        color='rgba(0, 0, 255, 0.4)',
                        thickness=2,
                        width=4
                    )
                ))
            
            titre = f"Optimalité et Variance MCTS — Agents: {ag} | Objets: {ob} | Ratio: {rat} | Graine: {sd} | Pol: {pol}"
            
            y_min = max(0, df_filtre['Opti_Min_%'].min() - 5)
            y_max = max(105, df_filtre['Opti_Max_%'].max() + 5)
            
            fig.update_layout(
                title=titre,
                xaxis_title="Instances testées (Index)",
                yaxis_title="Pourcentage du score optimal (%)",
                template="plotly_white",
                hovermode="x unified",   # Permet de comparer facilement l'alignement sur un même index X
                yaxis=dict(range=[y_min, y_max]),
                height=500,
                margin=dict(l=40, r=40, t=60, b=40)
            )
            
            fig.show()

    dropdown_agents.observe(update_plot, names='value')
    dropdown_objets.observe(update_plot, names='value')
    dropdown_seed.observe(update_plot, names='value')
    dropdown_policy.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    
    display(ui, out)
    update_plot()

In [13]:
def plot_boxplot_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # Normalisation : on calcule l'optimalité en %
    df_safe = df[df['Solveur_Score'] > 0].copy()
    df_safe['Opti_Moyen_%'] = (df_safe['MCTS_Score_Moyen'] / df_safe['Solveur_Score']) * 100
    df_safe['Opti_Moyen_%'] = df_safe['Opti_Moyen_%'].clip(upper=100)

    # 1. Widgets
    dropdown_y = widgets.Dropdown(
        options=[('Optimalité du MCTS (%)', 'Opti'), ('Temps d\'exécution (µs)', 'Temps')],
        value='Temps',
        description='📈 Analyser:',
        layout={'width': 'max-content'}
    )
    
    axe_x_options = [('Nombre d\'Agents', 'Agents'), ('Nombre d\'Objets', 'Objets')]
    if 'Ratio' in df_safe.columns:
        axe_x_options.append(('Ratio Random', 'Ratio'))
        
    dropdown_x = widgets.Dropdown(
        options=axe_x_options,
        value='Agents',
        description='📊 Grouper par:',
        layout={'width': 'max-content'}
    )
    
    policy_options = ['Tous'] + sorted(df_safe['Policy'].unique().tolist()) if 'Policy' in df_safe.columns else ["Non défini"]
    dropdown_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})

    ui = widgets.HBox([dropdown_y, dropdown_x, dropdown_policy])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            col_y = dropdown_y.value
            col_x = dropdown_x.value
            pol = dropdown_policy.value
            
            df_plot = df_safe.copy()
            
            # Filtre dynamique de la Politique (uniquement si une politique précise est choisie)
            if pol != 'Tous' and pol != "Non défini": 
                df_plot = df_plot[df_plot['Policy'] == pol]
            
            if df_plot.empty:
                print("⚠️ Aucune donnée pour cette sélection.")
                return
            
            # Forcer les catégories pour l'axe X
            suffixe = dropdown_x.label.split(" ")[-1]
            df_plot['X_Labels'] = df_plot[col_x].astype(str) + " " + suffixe
            sorted_unique_vals = sorted(df_safe[col_x].unique())
            sorted_labels = [str(val) + " " + suffixe for val in sorted_unique_vals]
            df_plot['X_Labels'] = pd.Categorical(df_plot['X_Labels'], categories=sorted_labels, ordered=True)

            # --- CAS 1 : ANALYSE DE L'OPTIMALITÉ ---
            if col_y == 'Opti':
                # Si 'Tous' est sélectionné, on colore par Politique pour créer une boîte par politique
                couleur_axe = 'Policy' if pol == 'Tous' and 'Policy' in df_plot.columns else 'X_Labels'
                
                fig = px.box(
                    df_plot,
                    x='X_Labels',
                    y="Opti_Moyen_%",
                    color=couleur_axe,
                    title=f"Distribution de l'Optimalité (Pol: {pol})",
                    labels={'X_Labels': dropdown_x.label, "Opti_Moyen_%": "Optimalité (%)", "Policy": "Politique"},
                    points="all", 
                    hover_data=["Solveur_Score", "MCTS_Score_Moyen"] + (["Seed"] if "Seed" in df_plot.columns else []),
                    color_discrete_sequence=px.colors.qualitative.Prism if pol == 'Tous' else None
                )
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                y_min = max(0, df_plot['Opti_Moyen_%'].min() - 5)
                fig.update_layout(yaxis=dict(range=[y_min, 105]), boxmode="group")
                if pol != 'Tous':
                    fig.update_layout(showlegend=False)
                
            # --- CAS 2 : ANALYSE DES TEMPS D'EXÉCUTION ---
            else: 
                colonnes_a_garder = ['X_Labels', 'Policy'] if 'Policy' in df_plot.columns else ['X_Labels']
                if 'Seed' in df_plot.columns:
                    colonnes_a_garder.append('Seed')
                    
                df_melt = df_plot.melt(
                    id_vars=colonnes_a_garder,
                    value_vars=['Solveur_Temps_us', 'MCTS_Temps_Moyen_us'],
                    var_name='Algorithme',
                    value_name='Temps_us'
                )
                
                # Distinction dynamique des algorithmes et des politiques dans la légende
                if pol == 'Tous' and 'Policy' in df_melt.columns:
                    df_melt['Modèle'] = df_melt.apply(
                        lambda r: 'Solveur Exact' if r['Algorithme'] == 'Solveur_Temps_us' else f"MCTS ({r['Policy']})", 
                        axis=1
                    )
                else:
                    df_melt['Modèle'] = df_melt['Algorithme'].replace({'Solveur_Temps_us': 'Solveur Exact', 'MCTS_Temps_Moyen_us': f'MCTS ({pol})'})
                
                df_melt['Temps_us'] = df_melt['Temps_us'].clip(lower=1)
                
                fig = px.box(
                    df_melt,
                    x='X_Labels',
                    y="Temps_us",
                    color="Modèle",
                    title=f"Comparaison des Temps (Pol: {pol})",
                    labels={'X_Labels': dropdown_x.label, "Temps_us": "Temps d'exécution (µs)", "Modèle": "Configuration"},
                    points="outliers"
                )
                fig.update_yaxes(type="log") 
                fig.update_layout(boxmode="group") 

            fig.update_layout(template="plotly_white", margin=dict(t=60, b=40, l=40, r=40))
            fig.update_xaxes(type='category')
            fig.show()

    # Écouteurs d'événements
    dropdown_y.observe(update_plot, names='value')
    dropdown_x.observe(update_plot, names='value')
    dropdown_policy.observe(update_plot, names='value')
    
    display(ui, out)
    update_plot()

In [14]:
def plot_nuage_points_metriques_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Préparation et Normalisation des données
    df_plot = df[df['Solveur_Score'] > 0].copy()
    df_plot['Optimalite_MCTS_%'] = (df_plot['MCTS_Score_Moyen'] / df_plot['Solveur_Score']) * 100
    df_plot['Optimalite_MCTS_%'] = df_plot['Optimalite_MCTS_%'].clip(upper=100)

    # 2. Création des Profils Combinés (Signatures)
    def get_signature(row):
        sigs = []
        if row.get('MCTS_ParetoOptimal', 0) >= 0.5: 
            sigs.append('PO')
            
        if row.get('MCTS_EF', 0) >= 0.5: 
            sigs.append('EF')
        elif row.get('MCTS_EFX', 0) >= 0.5: 
            sigs.append('EFX')
        elif row.get('MCTS_EF1', 0) >= 0.5: 
            sigs.append('EF1')
            
        if row.get('MCTS_Prop', 0) >= 0.5: 
            sigs.append('Prop')
            
        return " + ".join(sigs) if sigs else "Aucune"

    df_plot['Profil_Equite'] = df_plot.apply(get_signature, axis=1)

    # 3. Listes des options pour les widgets
    agents_options = ['Tous'] + sorted(df_plot['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_plot['Objets'].unique().tolist())
    ratio_options = ['Tous'] + sorted(df_plot['Ratio'].unique().tolist()) if 'Ratio' in df_plot.columns else ["Non défini"]
    policy_options = ['Tous'] + sorted(df_plot['Policy'].unique().tolist()) if 'Policy' in df_plot.columns else ["Non défini"]
    
    axes_options = [
        ('Optimalité MCTS (%)', 'Optimalite_MCTS_%'),
        ('Score Optimal (Solveur)', 'Solveur_Score'),
        ('Score Moyen MCTS', 'MCTS_Score_Moyen'),
        ('Temps d\'exécution MCTS (µs)', 'MCTS_Temps_Moyen_us'),
        ('Ratio Random', 'Ratio')
    ]
    
    metriques_dispos = [m.replace('MCTS_', '') for m in df_plot.columns if m.startswith('MCTS_') and m.replace('MCTS_', '') in ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]]
    
    # AJOUT : Option pour colorer directement par Politique
    couleur_options = [
        ('💡 Combinaison globale (Profil)', 'Profil_Equite'),
        ('⚙️ Politique (Policy)', 'Policy')
    ] + [(f"Métrique seule : {m}", f"MCTS_{m}") for m in metriques_dispos]

    # 4. Création des widgets
    dd_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dd_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    slider_ratio = widgets.SelectionSlider(options=ratio_options, value='Tous', description='🎲 Ratio:', continuous_update=False, layout={'width': '350px'})
    dd_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'}) # NOUVEAU
    
    dd_x = widgets.Dropdown(options=axes_options, value='Solveur_Score', description='➡️ Axe X:', layout={'width': 'max-content'})
    dd_y = widgets.Dropdown(options=axes_options, value='Optimalite_MCTS_%', description='⬆️ Axe Y:', layout={'width': 'max-content'})
    dd_couleur = widgets.Dropdown(options=couleur_options, value='Profil_Equite', description='🎨 Couleur:', layout={'width': 'max-content'})

    ui_filtres = widgets.HBox([dd_agents, dd_objets, slider_ratio, dd_policy]) # Ajouté dd_policy ici
    ui_axes = widgets.HBox([dd_x, dd_y, dd_couleur])
    ui = widgets.VBox([ui_filtres, ui_axes])
    
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            # Filtrage des données
            df_filtre = df_plot.copy()
            if dd_agents.value != 'Tous': df_filtre = df_filtre[df_filtre['Agents'] == dd_agents.value]
            if dd_objets.value != 'Tous': df_filtre = df_filtre[df_filtre['Objets'] == dd_objets.value]
            if slider_ratio.value != 'Tous' and slider_ratio.value != "Non défini": df_filtre = df_filtre[df_filtre['Ratio'] == slider_ratio.value]
            if dd_policy.value != 'Tous' and dd_policy.value != "Non défini": df_filtre = df_filtre[df_filtre['Policy'] == dd_policy.value] # NOUVEAU

            if df_filtre.empty:
                print("⚠️ Aucune donnée pour cette sélection.")
                return

            col_x = dd_x.value
            col_y = dd_y.value
            col_c = dd_couleur.value

            # Configuration des infobulles (Hover Data)
            hover_dict = { 
                col_x: ':.2f', col_y: ':.2f',
                "Agents": True, "Objets": True, "Ratio": True, "Policy": True, # Inclus par défaut
                "Profil_Equite": True
            }
            for m in metriques_dispos:
                hover_dict[f"MCTS_{m}"] = ':.2f'

            # --- DESSIN DU GRAPHIQUE ---
            if col_c in ['Profil_Equite', 'Policy']:
                # Mode Combinaison ou Politique (Couleurs discrètes)
                fig = px.scatter(df_filtre, x=col_x, y=col_y, color=col_c,
                    title=f"Analyse Multidimensionnelle (Filtre Pol: {dd_policy.value})",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_c: "Groupe (Couleur)"},
                    hover_data=hover_dict,
                    opacity=0.8,
                    color_discrete_sequence=px.colors.qualitative.Prism
                )
            else:
                # Mode Métrique Unique (Dégradé de couleur continu Rouge -> Vert)
                fig = px.scatter(df_filtre, x=col_x, y=col_y, color=col_c,
                    title=f"Réussite de la métrique {col_c.replace('MCTS_', '')} par le MCTS",
                    labels={col_x: dd_x.label, col_y: dd_y.label, col_c: "Taux de réussite (0 à 1)"},
                    hover_data=hover_dict,
                    color_continuous_scale="RdYlGn",
                    range_color=[0, 1],
                    opacity=0.8
                )

            fig.update_traces(marker=dict(size=9, line=dict(width=1, color='DarkSlateGrey')))
            
            # Ligne de référence si l'optimalité est sur l'axe Y
            if col_y == 'Optimalite_MCTS_%':
                fig.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="Optimum (100%)")
                fig.update_layout(yaxis=dict(range=[max(0, df_filtre[col_y].min() - 5), 105]))

            if 'Temps' in col_y: fig.update_yaxes(type="log")
            if 'Temps' in col_x: fig.update_xaxes(type="log")

            fig.update_layout(template="plotly_white", margin=dict(t=60, b=40, l=40, r=40), height=600)
            fig.show()

    # 5. Liaisons
    dd_agents.observe(update_plot, names='value')
    dd_objets.observe(update_plot, names='value')
    slider_ratio.observe(update_plot, names='value')
    dd_policy.observe(update_plot, names='value') # NOUVEAU
    dd_x.observe(update_plot, names='value')
    dd_y.observe(update_plot, names='value')
    dd_couleur.observe(update_plot, names='value')

    display(ui, out)
    update_plot()

In [15]:
def plot_nuage_points_3d_interactive(df):
    if df.empty:
        print("Aucune donnée trouvée.")
        return

    # 1. Préparation et Normalisation des données
    df_plot = df[df['Solveur_Score'] > 0].copy()
    df_plot['Optimalite_MCTS_%'] = (df_plot['MCTS_Score_Moyen'] / df_plot['Solveur_Score']) * 100
    df_plot['Optimalite_MCTS_%'] = df_plot['Optimalite_MCTS_%'].clip(upper=100)

    # Création des Profils d'Équité (Signatures)
    def get_signature(row):
        sigs = []
        if row.get('MCTS_ParetoOptimal', 0) >= 0.5: sigs.append('PO')
        if row.get('MCTS_EF', 0) >= 0.5: sigs.append('EF')
        elif row.get('MCTS_EFX', 0) >= 0.5: sigs.append('EFX')
        elif row.get('MCTS_EF1', 0) >= 0.5: sigs.append('EF1')
        if row.get('MCTS_Prop', 0) >= 0.5: sigs.append('Prop')
        return " + ".join(sigs) if sigs else "Aucune"
        
    df_plot['Profil_Equite'] = df_plot.apply(get_signature, axis=1)

    # 2. Options des widgets
    agents_options = ['Tous'] + sorted(df_plot['Agents'].unique().tolist())
    objets_options = ['Tous'] + sorted(df_plot['Objets'].unique().tolist())
    policy_options = ['Tous'] + sorted(df_plot['Policy'].unique().tolist()) if 'Policy' in df_plot.columns else ["Non défini"]
    
    axes_options = [
        ('Optimalité MCTS (%)', 'Optimalite_MCTS_%'),
        ('Score Optimal (Solveur)', 'Solveur_Score'),
        ('Score Moyen MCTS', 'MCTS_Score_Moyen'),
        ("Temps d'exécution MCTS (µs)", 'MCTS_Temps_Moyen_us'),
        ("Ratio Random", 'Ratio')
    ]
    
    metriques_dispos = [m.replace('MCTS_', '') for m in df_plot.columns if m.startswith('MCTS_') and m.replace('MCTS_', '') in ["ParetoOptimal", "Prop", "EF", "EFX", "EF1"]]
    couleur_options = [('💡 Combinaison globale (Profil)', 'Profil_Equite'), ('⚙️ Politique (Policy)', 'Policy')] + [(f"Métrique seule : {m}", f"MCTS_{m}") for m in metriques_dispos]

    # 3. Création des widgets
    dd_agents = widgets.Dropdown(options=agents_options, value='Tous', description='👥 Agents:', layout={'width': 'max-content'})
    dd_objets = widgets.Dropdown(options=objets_options, value='Tous', description='📦 Objets:', layout={'width': 'max-content'})
    dd_policy = widgets.Dropdown(options=policy_options, value='Tous', description='⚙️ Politique:', layout={'width': 'max-content'})
    
    dd_x = widgets.Dropdown(options=axes_options, value='Solveur_Score', description='➡️ Axe X:', layout={'width': 'max-content'})
    dd_y = widgets.Dropdown(options=axes_options, value='Optimalite_MCTS_%', description='⬆️ Axe Y:', layout={'width': 'max-content'})
    dd_z = widgets.Dropdown(options=axes_options, value='Ratio', description='↗️ Axe Z:', layout={'width': 'max-content'})
    dd_couleur = widgets.Dropdown(options=couleur_options, value='Profil_Equite', description='🎨 Couleur:', layout={'width': 'max-content'})

    ui = widgets.VBox([
        widgets.HBox([dd_agents, dd_objets, dd_policy]),
        widgets.HBox([dd_x, dd_y, dd_z]),
        widgets.HBox([dd_couleur])
    ])
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            out.clear_output(wait=True)
            
            # Filtrage dynamique de la copie
            df_filtre = df_plot.copy()
            if dd_agents.value != 'Tous': 
                df_filtre = df_filtre[df_filtre['Agents'] == dd_agents.value]
            if dd_objets.value != 'Tous': 
                df_filtre = df_filtre[df_filtre['Objets'] == dd_objets.value]
            if dd_policy.value != 'Tous' and dd_policy.value != "Non défini": 
                df_filtre = df_filtre[df_filtre['Policy'] == dd_policy.value]

            if df_filtre.empty:
                print("⚠️ Aucune donnée pour cette sélection.")
                return

            col_x = dd_x.value
            col_y = dd_y.value
            col_z = dd_z.value
            col_c = dd_couleur.value

            # Sécurité anti-crash Plotly : Conversion explicite en string ou float selon la nature de la colonne
            for col in [col_x, col_y, col_z]:
                if df_filtre[col].dtype == 'object':
                    df_filtre[col] = df_filtre[col].astype(str)
                else:
                    df_filtre[col] = pd.to_numeric(df_filtre[col], errors='coerce')

            # Configuration des infobulles (Hover Data)
            hover_dict = {
                col_x: True, col_y: True, col_z: True,
                "Agents": True, "Objets": True, "Ratio": True, "Policy": True
            }

            # --- DESSIN DU GRAPHIQUE 3D ---
            try:
                if col_c in ['Profil_Equite', 'Policy']:
                    # Mode qualitatif (Palette discrète de couleurs)
                    df_filtre[col_c] = df_filtre[col_c].astype(str) # Forcer le format texte pour la légende
                    fig = px.scatter_3d(
                        df_filtre, x=col_x, y=col_y, z=col_z, 
                        color=col_c, opacity=0.8, 
                        title=f"Visualisation 3D — Focus Politique: {dd_policy.value}",
                        labels={col_x: dd_x.label, col_y: dd_y.label, col_z: dd_z.label, col_c: "Légende"},
                        hover_data=hover_dict,
                        color_discrete_sequence=px.colors.qualitative.Prism
                    )
                else:
                    # Mode quantitatif (Dégradé continu de couleurs Rouge -> Vert)
                    df_filtre[col_c] = pd.to_numeric(df_filtre[col_c], errors='coerce')
                    fig = px.scatter_3d(
                        df_filtre, x=col_x, y=col_y, z=col_z, 
                        color=col_c, opacity=0.8,
                        title=f"Visualisation 3D — Taux métrique {col_c.replace('MCTS_', '')}",
                        labels={col_x: dd_x.label, col_y: dd_y.label, col_z: dd_z.label, col_c: "Taux (0 à 1)"},
                        hover_data=hover_dict,
                        color_continuous_scale="RdYlGn", range_color=[0, 1]
                    )

                # Style des marqueurs
                fig.update_traces(marker=dict(size=4))
                
                # Gestion des axes logarithmiques pour le Temps
                log_settings = {}
                if 'Temps' in dd_x.label: log_settings['xaxis_type'] = "log"
                if 'Temps' in dd_y.label: log_settings['yaxis_type'] = "log"
                if 'Temps' in dd_z.label: log_settings['zaxis_type'] = "log"
                if log_settings:
                    fig.update_scenes(**log_settings)

                fig.update_layout(template="plotly_white", margin=dict(t=40, b=0, l=0, r=0), height=700)
                fig.show()
                
            except Exception as e:
                print(f"❌ Erreur lors de la génération du graphique 3D Plotly : {e}")
                print("Vérifiez que votre navigateur ou votre notebook Jupyter supporte WebGL.")

    # 4. Liaisons des événements
    dd_agents.observe(update_plot, names='value')
    dd_objets.observe(update_plot, names='value')
    dd_policy.observe(update_plot, names='value')
    dd_x.observe(update_plot, names='value')
    dd_y.observe(update_plot, names='value')
    dd_z.observe(update_plot, names='value')
    dd_couleur.observe(update_plot, names='value')

    display(ui, out)
    update_plot()

In [25]:
# Exécution
nom_du_fichier_zip = "../" + "results/experiments_30-06-2026_13-17-06.zip"
df = parse_zip_experiments(nom_du_fichier_zip)

100%|██████████| 2605/2605 [00:00<00:00, 10230.41it/s]


In [17]:
#plot_comparisons(df)

In [18]:
#plot_impact_ratio(df)

In [19]:
#plot_ratio_impact_on_score(df)

In [20]:
#plot_erreur_normalisee(df)

In [27]:

""" 
plot_comparisons_interactive(df)
print("---")
plot_impact_ratio_interactive(df)
print("---")
plot_ratio_impact_on_score_interactive(df)
print("---")
plot_erreur_normalisee_interactive(df)
print("---")
plot_metrics_comparison_interactive(df) 
print("---")
widget_variance_mcts_normalise(df)
print("---") 
plot_boxplot_interactive(df) 
print("---") 
plot_nuage_points_metriques_interactive(df)
print("---") 
"""
plot_nuage_points_3d_interactive(df)

Output()